# Level 5 Measured comparison

Inspect the retained RTL evidence and replay one workload with identical geometry.

In [1]:
from pathlib import Path
import sys, json, csv, subprocess
cwd=Path.cwd().resolve()
ROOT=next(p for b in [cwd,*cwd.parents] for p in [b,b/'Phase1'] if (p/'level4/rtl/ai_cache.sv').is_file())
print('Phase 1 source tree located')

Phase 1 source tree located


In [2]:
with (ROOT/'level5/results/reference/summary.csv').open() as f: rows=list(csv.DictReader(f))
for r in rows:
    if r['workload'] in ['column_stride','pollution_return','modelled_matmul_ijk_ab_reads']:
        print(r['workload'],r['policy'],'cycles=',r['completion_cycles'],'speedup=',round(float(r['speedup']),3),'DRAM=',r['dram_reads'])

column_stride none cycles= 21098 speedup= 1.0 DRAM= 512
column_stride next_line cycles= 21098 speedup= 1.0 DRAM= 1024
column_stride stride cycles= 12573 speedup= 1.678 DRAM= 513
modelled_matmul_ijk_ab_reads none cycles= 6940 speedup= 1.0 DRAM= 92
modelled_matmul_ijk_ab_reads next_line cycles= 7154 speedup= 0.97 DRAM= 205
modelled_matmul_ijk_ab_reads stride cycles= 6940 speedup= 1.0 DRAM= 92
pollution_return none cycles= 7888 speedup= 1.0 DRAM= 4
pollution_return next_line cycles= 8895 speedup= 0.887 DRAM= 133
pollution_return stride cycles= 8927 speedup= 0.884 DRAM= 132


## Correct evaluation
Demand hit rate = hits / accepted demand accesses. Speedup = no-prefetch completion cycles / policy completion cycles. A late prediction can shorten a miss without creating a hit.

In [3]:
cmd=[sys.executable,'level5/scripts/run_benchmarks.py','--output','build/notebooks/level5','--workload','column_stride']
run=subprocess.run(cmd,cwd=ROOT,text=True,capture_output=True)
print(run.stdout)
if run.returncode:raise RuntimeError(run.stderr)

PASS column_stride                     none      hits=    0/512 completion=  21098 DRAM=512
PASS column_stride                     next_line hits=    0/512 completion=  21098 DRAM=1024
PASS column_stride                     stride    hits=  509/512 completion=  12573 DRAM=513
Saved 3 measured runs and 1536 checked responses to build/notebooks/level5



## Interpret negative cases
Pollution causes additional demand misses. Filtered matrix A/B traces show no stride gain. Preserve these results when developing Phase 2.